# PC5 v4 - YOLO Segmentation Training

This notebook trains a YOLO segmentation model on the curated PlantVillage PC5 sample. The dataset has no human segmentation masks, so the notebook creates pseudo-labels with classical foreground segmentation and documents that quantitative metrics are measured against pseudo-labels, not ground truth disease masks.


## Setup and imports
Load libraries, install Ultralytics only if missing, and initialize deterministic settings.


In [ ]:
import importlib.util
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import traceback
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})


## Paths and progress tracking
Resolve Kaggle input paths and create output folders requested by the assignment.


In [ ]:
INPUT_ROOT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")

CATEGORIES = [
    "healthy_leaves",
    "small_diseased_regions",
    "large_diseased_regions",
    "simple_backgrounds",
    "complex_backgrounds",
    "multiple_leaves",
    "partial_occlusions",
]

DATASET_CANDIDATES = [
    INPUT_ROOT / "cv-pc5-v3-plantvillage-segmentation-data",
    INPUT_ROOT / "datasets" / "jeffreyamc" / "cv-pc5-v3-plantvillage-segmentation-data",
]
for candidate in INPUT_ROOT.glob("**"):
    if candidate.is_dir() and all((candidate / category).exists() for category in CATEGORIES):
        DATASET_CANDIDATES.append(candidate)

DATASET_SLUG_DIR = next((p for p in DATASET_CANDIDATES if p.exists() and all((p / c).exists() for c in CATEGORIES)), None)

PREPARED_DIR = WORKING_DIR / "prepared_yolo_dataset"
PSEUDO_PREVIEW_DIR = WORKING_DIR / "pseudo_labels_preview"
QUAL_DIR = WORKING_DIR / "qualitative_results"
QUAL_BY_CATEGORY_DIR = QUAL_DIR / "by_category"
FAILURES_DIR = QUAL_DIR / "failures"
QUANT_DIR = WORKING_DIR / "quantitative_metrics"
RUNS_DIR = WORKING_DIR / "runs"
SUMMARY_PATH = WORKING_DIR / "run_summary.json"

for path in [PREPARED_DIR, PSEUDO_PREVIEW_DIR, QUAL_BY_CATEGORY_DIR, FAILURES_DIR, QUANT_DIR, RUNS_DIR]:
    path.mkdir(parents=True, exist_ok=True)
for category in CATEGORIES:
    (QUAL_BY_CATEGORY_DIR / category).mkdir(parents=True, exist_ok=True)

progress = {
    "status": "running",
    "stage": "start",
    "note": "YOLO segmentation trained with pseudo-labels generated from classical leaf foreground masks.",
    "artifacts": [],
}

def save_progress():
    SUMMARY_PATH.write_text(json.dumps(progress, indent=2), encoding="utf-8")

save_progress()
print({"working_dir": str(WORKING_DIR), "dataset_dir": str(DATASET_SLUG_DIR)})


## Dataset inventory
Read the seven PC5 categories and build a balanced training subset to keep Kaggle T4 runtime manageable.


In [ ]:
progress["stage"] = "dataset_inventory"
save_progress()

if DATASET_SLUG_DIR is None:
    visible = []
    for path in sorted(INPUT_ROOT.glob("*")):
        visible.append(str(path))
    for path in sorted((INPUT_ROOT / "datasets").glob("*/*")) if (INPUT_ROOT / "datasets").exists() else []:
        visible.append(str(path))
    raise FileNotFoundError(f"Could not find PC5 category folders under /kaggle/input. Visible roots: {visible}")

records = []
for category in CATEGORIES:
    category_dir = DATASET_SLUG_DIR / category
    if not category_dir.exists():
        raise FileNotFoundError(f"Missing category directory: {category_dir}")
    images = sorted([p for p in category_dir.glob("*.jpg")])
    for path in images:
        records.append({"category": category, "path": path, "filename": path.name})

inventory_df = pd.DataFrame(records)
counts = inventory_df.groupby("category").size().to_dict()
print({"total_images": len(inventory_df), "category_counts": counts})

# Use 120 per category max: enough for visible learning, bounded for Kaggle class runtime.
MAX_PER_CATEGORY = 120
sampled = []
for category in CATEGORIES:
    cat_df = inventory_df[inventory_df["category"] == category].sort_values("filename")
    sampled.append(cat_df.head(MAX_PER_CATEGORY))
work_df = pd.concat(sampled, ignore_index=True)
work_df["row_id"] = [f"img_{i:05d}" for i in range(len(work_df))]

# Deterministic split within each category.
splits = []
for category, cat_df in work_df.groupby("category", sort=False):
    cat_df = cat_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    n_val = max(8, int(round(len(cat_df) * 0.2)))
    cat_df.loc[: n_val - 1, "split"] = "val"
    cat_df.loc[n_val:, "split"] = "train"
    splits.append(cat_df)
work_df = pd.concat(splits, ignore_index=True).sort_values("row_id")

inventory_out = QUANT_DIR / "training_inventory.csv"
work_df.assign(path=work_df["path"].astype(str)).to_csv(inventory_out, index=False)
progress["artifacts"].append(str(inventory_out))
save_progress()

print({"selected_images": len(work_df), "split_counts": work_df["split"].value_counts().to_dict()})


## Pseudo-label functions
Create binary leaf foreground masks, clean them morphologically, convert largest components to YOLO segmentation polygons, and flag weak masks.


In [ ]:
def read_rgb(path):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"Could not read image: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def leaf_foreground_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    h, s, v = cv2.split(hsv)
    l, a, b = cv2.split(lab)

    # PlantVillage often has gray/simple backgrounds. Combine green/yellow color evidence,
    # saturation, and Otsu thresholding to get a leaf foreground pseudo-mask.
    green = ((h >= 20) & (h <= 95) & (s >= 25) & (v >= 35)).astype(np.uint8) * 255
    sat = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    chroma = cv2.threshold(cv2.absdiff(a, b), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    mask = cv2.bitwise_or(green, cv2.bitwise_and(sat, chroma))

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.medianBlur(mask, 5)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), 8)
    if num_labels <= 1:
        return np.zeros(mask.shape, dtype=np.uint8)

    h_img, w_img = mask.shape
    min_area = max(64, int(0.01 * h_img * w_img))
    clean = np.zeros(mask.shape, dtype=np.uint8)
    areas = stats[1:, cv2.CC_STAT_AREA]
    order = np.argsort(areas)[::-1] + 1
    for label in order[:3]:
        area = stats[label, cv2.CC_STAT_AREA]
        if area >= min_area:
            clean[labels == label] = 255

    if clean.sum() == 0:
        label = int(order[0])
        clean[labels == label] = 255
    return clean


def mask_to_yolo_polygons(mask, min_area_ratio=0.005, epsilon_ratio=0.003):
    h, w = mask.shape
    contours, _ = cv2.findContours((mask > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    lines = []
    polygons_px = []
    min_area = h * w * min_area_ratio
    for contour in contours[:3]:
        area = cv2.contourArea(contour)
        if area < min_area:
            continue
        epsilon = max(1.0, epsilon_ratio * cv2.arcLength(contour, True))
        approx = cv2.approxPolyDP(contour, epsilon, True).reshape(-1, 2)
        if len(approx) < 3:
            continue
        coords = []
        for x, y in approx:
            coords.extend([float(np.clip(x / w, 0, 1)), float(np.clip(y / h, 0, 1))])
        lines.append("0 " + " ".join(f"{value:.6f}" for value in coords))
        polygons_px.append(approx)
    return lines, polygons_px


def overlay_mask(rgb, mask, color=(30, 200, 80), alpha=0.45):
    overlay = rgb.copy()
    color_arr = np.array(color, dtype=np.uint8)
    overlay[mask > 0] = (overlay[mask > 0] * (1 - alpha) + color_arr * alpha).astype(np.uint8)
    edges = cv2.Canny((mask > 0).astype(np.uint8) * 255, 50, 150)
    overlay[edges > 0] = np.array([255, 40, 40], dtype=np.uint8)
    return overlay


def write_image(path, rgb):
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(rgb).save(path, quality=95)

print("Pseudo-label helpers ready")


## Build YOLO segmentation dataset
Write images, YOLO polygon labels, previews, and mask quality flags.


In [ ]:
progress["stage"] = "prepare_yolo_dataset"
save_progress()

for split in ["train", "val"]:
    (PREPARED_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (PREPARED_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

label_rows = []
failure_rows = []
preview_limit_per_category = 10
preview_counts = Counter()

for row in work_df.itertuples(index=False):
    rgb = read_rgb(row.path)
    mask = leaf_foreground_mask(rgb)
    lines, polygons = mask_to_yolo_polygons(mask)
    h, w = mask.shape
    mask_area_ratio = float((mask > 0).mean())
    component_count, _, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), 8)
    fg_components = int(max(0, component_count - 1))

    out_name = f"{row.row_id}__{row.category}__{Path(row.path).stem}.jpg"
    image_out = PREPARED_DIR / "images" / row.split / out_name
    label_out = PREPARED_DIR / "labels" / row.split / (Path(out_name).stem + ".txt")
    shutil.copy2(row.path, image_out)
    label_out.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")

    weak_reasons = []
    if not lines:
        weak_reasons.append("empty_polygon")
    if mask_area_ratio < 0.03:
        weak_reasons.append("small_mask_area")
    if mask_area_ratio > 0.90:
        weak_reasons.append("large_mask_area")
    if fg_components > 4:
        weak_reasons.append("many_components")

    if preview_counts[row.category] < preview_limit_per_category:
        preview_path = PSEUDO_PREVIEW_DIR / row.category / f"{row.row_id}_pseudo_overlay.jpg"
        write_image(preview_path, overlay_mask(rgb, mask))
        preview_counts[row.category] += 1

    if weak_reasons:
        failure_path = FAILURES_DIR / f"pseudo_{row.row_id}_{'_'.join(weak_reasons)}.jpg"
        write_image(failure_path, overlay_mask(rgb, mask, color=(255, 120, 30)))
        failure_rows.append({
            "row_id": row.row_id,
            "category": row.category,
            "split": row.split,
            "path": str(row.path),
            "mask_area_ratio": mask_area_ratio,
            "component_count": fg_components,
            "reasons": ";".join(weak_reasons),
            "failure_preview": str(failure_path),
        })

    label_rows.append({
        "row_id": row.row_id,
        "category": row.category,
        "split": row.split,
        "source_path": str(row.path),
        "image_path": str(image_out),
        "label_path": str(label_out),
        "polygon_count": len(lines),
        "mask_area_ratio": mask_area_ratio,
        "component_count": fg_components,
        "weak_reasons": ";".join(weak_reasons),
    })

labels_df = pd.DataFrame(label_rows)
failures_df = pd.DataFrame(failure_rows)
labels_csv = QUANT_DIR / "pseudo_label_inventory.csv"
failures_csv = QUANT_DIR / "pseudo_label_failures.csv"
labels_df.to_csv(labels_csv, index=False)
failures_df.to_csv(failures_csv, index=False)

# YOLO data config. Single class means foreground leaf pseudo-mask.
data_yaml = PREPARED_DIR / "data.yaml"
data_yaml.write_text(
    f"path: {PREPARED_DIR}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: leaf_foreground_pseudo_mask\n",
    encoding="utf-8",
)

progress["artifacts"].extend([str(labels_csv), str(failures_csv), str(data_yaml), str(PSEUDO_PREVIEW_DIR)])
save_progress()
print({
    "labels": len(labels_df),
    "empty_labels": int((labels_df["polygon_count"] == 0).sum()),
    "weak_labels": int(labels_df["weak_reasons"].astype(bool).sum()),
    "data_yaml": str(data_yaml),
})


## Train YOLO segmentation
Fine-tune `yolo11n-seg.pt` with a bounded training budget for Kaggle T4.


In [ ]:
progress["stage"] = "train_yolo"
save_progress()

model = YOLO("yolo11n-seg.pt")
train_result = model.train(
    data=str(PREPARED_DIR / "data.yaml"),
    epochs=25,
    imgsz=640,
    batch=16,
    workers=2,
    seed=SEED,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(RUNS_DIR),
    name="yolo11n_seg_leaf_pseudolabels",
    exist_ok=True,
    patience=8,
    verbose=False,
)

train_dir = Path(getattr(train_result, "save_dir", RUNS_DIR / "yolo11n_seg_leaf_pseudolabels"))
best_src = train_dir / "weights" / "best.pt"
last_src = train_dir / "weights" / "last.pt"
best_out = WORKING_DIR / "best_model.pt"
if best_src.exists():
    shutil.copy2(best_src, best_out)
elif last_src.exists():
    shutil.copy2(last_src, best_out)

progress["train_dir"] = str(train_dir)
progress["artifacts"].extend([str(train_dir), str(best_out)])
save_progress()
print({"train_dir": str(train_dir), "best_model_exists": best_out.exists()})


## Quantitative validation
Run YOLO validation against pseudo-labels and collect mask precision/recall/mAP where available.


In [ ]:
progress["stage"] = "validate_yolo"
save_progress()

trained_model_path = WORKING_DIR / "best_model.pt"
trained_model = YOLO(str(trained_model_path if trained_model_path.exists() else "yolo11n-seg.pt"))
metrics = trained_model.val(
    data=str(PREPARED_DIR / "data.yaml"),
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(RUNS_DIR),
    name="validation",
    exist_ok=True,
    verbose=False,
)

metric_values = {}
for attr in ["box", "seg"]:
    metric_obj = getattr(metrics, attr, None)
    if metric_obj is not None:
        for key in ["mp", "mr", "map50", "map", "map75"]:
            value = getattr(metric_obj, key, None)
            if value is not None:
                try:
                    metric_values[f"{attr}_{key}"] = float(value)
                except Exception:
                    pass

metrics_path = QUANT_DIR / "yolo_validation_metrics.json"
metrics_path.write_text(json.dumps(metric_values, indent=2), encoding="utf-8")
progress["artifacts"].append(str(metrics_path))
save_progress()
print(metric_values)


## Inference metrics helpers
Convert predicted polygons back to masks to estimate IoU against pseudo-labels and measure inference speed.


In [ ]:
def yolo_label_to_mask(label_path, shape):
    h, w = shape
    mask = np.zeros((h, w), dtype=np.uint8)
    if not Path(label_path).exists():
        return mask
    for line in Path(label_path).read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        vals = [float(v) for v in parts[1:]]
        pts = []
        for x, y in zip(vals[0::2], vals[1::2]):
            pts.append([int(round(x * w)), int(round(y * h))])
        if len(pts) >= 3:
            cv2.fillPoly(mask, [np.array(pts, dtype=np.int32)], 255)
    return mask


def result_to_mask(result, shape):
    h, w = shape
    mask = np.zeros((h, w), dtype=np.uint8)
    masks = getattr(result, "masks", None)
    if masks is None or masks.xy is None:
        return mask
    for poly in masks.xy:
        if poly is None or len(poly) < 3:
            continue
        pts = np.array(poly, dtype=np.int32)
        cv2.fillPoly(mask, [pts], 255)
    return mask


def mask_iou(a, b):
    aa = a > 0
    bb = b > 0
    union = np.logical_or(aa, bb).sum()
    if union == 0:
        return 1.0
    return float(np.logical_and(aa, bb).sum() / union)


def side_by_side_overlay(rgb, pseudo_mask, pred_mask):
    pseudo = overlay_mask(rgb, pseudo_mask, color=(30, 200, 80))
    pred = overlay_mask(rgb, pred_mask, color=(50, 120, 255))
    return np.concatenate([rgb, pseudo, pred], axis=1)

print("Inference helpers ready")


## Qualitative and quantitative results
Run inference on validation images, save category folders, failure examples, speed metrics, and per-category IoU summaries.


In [ ]:
progress["stage"] = "qualitative_quantitative_results"
save_progress()

val_df = labels_df[labels_df["split"] == "val"].copy().reset_index(drop=True)
# Limit qualitative generation but keep all val rows for speed/IoU metrics if feasible.
MAX_INFERENCE_IMAGES = min(len(val_df), 140)
eval_df = val_df.head(MAX_INFERENCE_IMAGES).copy()

inference_rows = []
for idx, row in enumerate(eval_df.itertuples(index=False), start=1):
    rgb = read_rgb(row.source_path)
    h, w = rgb.shape[:2]
    pseudo_mask = yolo_label_to_mask(row.label_path, (h, w))

    t0 = time.perf_counter()
    results = trained_model.predict(
        source=rgb,
        imgsz=640,
        device=0 if torch.cuda.is_available() else "cpu",
        conf=0.25,
        verbose=False,
    )
    elapsed = time.perf_counter() - t0
    result = results[0]
    pred_mask = result_to_mask(result, (h, w))
    iou = mask_iou(pseudo_mask, pred_mask)
    pred_empty = int((pred_mask > 0).sum() == 0)
    pred_area_ratio = float((pred_mask > 0).mean())

    qualitative_img = side_by_side_overlay(rgb, pseudo_mask, pred_mask)
    qual_path = QUAL_BY_CATEGORY_DIR / row.category / f"{row.row_id}_orig_pseudo_pred.jpg"
    write_image(qual_path, qualitative_img)

    failure_reason = None
    if pred_empty:
        failure_reason = "empty_prediction"
    elif iou < 0.35:
        failure_reason = "low_iou"
    elif pred_area_ratio < 0.03:
        failure_reason = "small_prediction"
    elif pred_area_ratio > 0.90:
        failure_reason = "large_prediction"
    if failure_reason:
        failure_path = FAILURES_DIR / f"pred_{row.row_id}_{failure_reason}.jpg"
        write_image(failure_path, qualitative_img)
    else:
        failure_path = ""

    inference_rows.append({
        "row_id": row.row_id,
        "category": row.category,
        "source_path": row.source_path,
        "inference_time_sec": elapsed,
        "pseudo_mask_area_ratio": float((pseudo_mask > 0).mean()),
        "pred_mask_area_ratio": pred_area_ratio,
        "iou_vs_pseudo_mask": iou,
        "empty_prediction": pred_empty,
        "qualitative_path": str(qual_path),
        "failure_reason": failure_reason or "",
        "failure_path": str(failure_path),
    })

inference_df = pd.DataFrame(inference_rows)
inference_csv = QUANT_DIR / "inference_times.csv"
inference_df.to_csv(inference_csv, index=False)

per_category = inference_df.groupby("category").agg(
    image_count=("row_id", "count"),
    mean_inference_time_sec=("inference_time_sec", "mean"),
    median_inference_time_sec=("inference_time_sec", "median"),
    mean_iou_vs_pseudo_mask=("iou_vs_pseudo_mask", "mean"),
    empty_prediction_count=("empty_prediction", "sum"),
    mean_pseudo_mask_area_ratio=("pseudo_mask_area_ratio", "mean"),
    mean_pred_mask_area_ratio=("pred_mask_area_ratio", "mean"),
).reset_index()
per_category_csv = QUANT_DIR / "per_category_metrics.csv"
per_category.to_csv(per_category_csv, index=False)

metrics_summary = {
    "status": "complete",
    "pseudo_label_warning": "Metrics compare predictions against generated leaf foreground pseudo-labels, not human ground-truth lesion masks.",
    "selected_training_images": int(len(labels_df)),
    "train_images": int((labels_df["split"] == "train").sum()),
    "val_images": int((labels_df["split"] == "val").sum()),
    "evaluated_inference_images": int(len(inference_df)),
    "mean_inference_time_sec": float(inference_df["inference_time_sec"].mean()) if len(inference_df) else None,
    "median_inference_time_sec": float(inference_df["inference_time_sec"].median()) if len(inference_df) else None,
    "empty_prediction_count": int(inference_df["empty_prediction"].sum()) if len(inference_df) else 0,
    "mean_iou_vs_pseudo_mask": float(inference_df["iou_vs_pseudo_mask"].mean()) if len(inference_df) else None,
    "yolo_validation_metrics": metric_values,
    "per_category": per_category.to_dict(orient="records"),
}
metrics_summary_path = QUANT_DIR / "metrics_summary.json"
metrics_summary_path.write_text(json.dumps(metrics_summary, indent=2), encoding="utf-8")

progress["artifacts"].extend([str(inference_csv), str(per_category_csv), str(metrics_summary_path), str(QUAL_DIR)])
save_progress()
print({
    "evaluated": len(inference_df),
    "mean_iou": metrics_summary["mean_iou_vs_pseudo_mask"],
    "mean_time_sec": metrics_summary["mean_inference_time_sec"],
    "empty_predictions": metrics_summary["empty_prediction_count"],
})


## Finalize outputs
Zip bulky visual folders and write final run summary for download.


In [ ]:
progress["stage"] = "finalize_outputs"
save_progress()

zip_targets = [
    (PSEUDO_PREVIEW_DIR, WORKING_DIR / "pseudo_labels_preview.zip"),
    (QUAL_DIR, WORKING_DIR / "qualitative_results.zip"),
    (PREPARED_DIR, WORKING_DIR / "prepared_yolo_dataset.zip"),
    (QUANT_DIR, WORKING_DIR / "quantitative_metrics.zip"),
]
for folder, zip_path in zip_targets:
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=folder)
    progress["artifacts"].append(str(zip_path))

# Keep top-level deliverables small so Kaggle CLI downloads all important files
# without paginating through hundreds of individual images.
for bulky_dir in [PREPARED_DIR, PSEUDO_PREVIEW_DIR, QUAL_DIR, RUNS_DIR]:
    if bulky_dir.exists():
        shutil.rmtree(bulky_dir)

progress["status"] = "complete"
progress["stage"] = "done"
progress["artifacts"] = sorted(set(progress["artifacts"]))
save_progress()

print(json.dumps({
    "status": progress["status"],
    "stage": progress["stage"],
    "artifact_count": len(progress["artifacts"]),
    "summary_path": str(SUMMARY_PATH),
}, indent=2))
